## API Exploration — Jikan REST API

Objetivos:
- Conhecer os endpoints relevantes.
- Entender o formato das respostas.
- Identificar entidades e relacionamentos.
- Compreender a paginação.
- Verificar limites da API.
- Definir quais dados farão parte do MVP.

### Rate limit
- 3 req/s
- 60 req/min

Decisão
A implementação utilizará atraso entre requisições.

In [21]:
import requests
from pprint import pprint

In [22]:
BASE_URL = "https://api.jikan.moe/v4"

### Primeira requisição

Objetivo:

Verificar se a API responde corretamente.

In [23]:
response = requests.get(f"{BASE_URL}/top/anime")
response.status_code

200

### Estrutura da resposta

In [24]:
response.json().keys()

dict_keys(['pagination', 'data', 'links', 'meta'])

In [25]:
pprint(response.json()["pagination"])

{'current_page': 1,
 'has_next_page': True,
 'items': {'count': 25, 'per_page': 25, 'total': 30374},
 'last_visible_page': 1215}


### Paginação

Objetivos:

- quantidade por página
- total de páginas
- como detectar próxima página

In [26]:
pagination = response.json()["pagination"]

pagination

{'last_visible_page': 1215,
 'has_next_page': True,
 'current_page': 1,
 'items': {'count': 25, 'total': 30374, 'per_page': 25}}

Conclusões

- 25 registros por página.
- Campo has_next_page indica continuidade.
- Campo last_visible_page informa o limite.

### Estrutura de um anime

In [27]:
anime = response.json()["data"][0]

anime.keys()

dict_keys(['mal_id', 'url', 'images', 'trailer', 'approved', 'titles', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'type', 'source', 'episodes', 'status', 'airing', 'aired', 'duration', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites', 'synopsis', 'background', 'season', 'year', 'broadcast', 'producers', 'licensors', 'studios', 'genres', 'explicit_genres', 'themes', 'demographics'])

In [28]:
response = requests.get("https://api.jikan.moe/v4/top/anime?page=1")

print(type(response.json()))
print(response.json().keys())
print(response.status_code)

<class 'dict'>
dict_keys(['pagination', 'data'])
200


In [29]:
# pprint(response.json())

In [30]:
data = response.json()["data"]

print(type(data))
print(len(data))

<class 'list'>
25


In [31]:
anime = data[0]

print(type(anime))
print(anime.keys())

<class 'dict'>
dict_keys(['mal_id', 'url', 'images', 'trailer', 'approved', 'titles', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'type', 'source', 'episodes', 'status', 'airing', 'aired', 'duration', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites', 'synopsis', 'background', 'season', 'year', 'broadcast', 'producers', 'licensors', 'studios', 'genres', 'explicit_genres', 'themes', 'demographics'])


In [32]:
pprint(anime)

{'aired': {'from': '2023-09-29T00:00:00+00:00',
           'prop': {'from': {'day': 29, 'month': 9, 'year': 2023},
                    'to': {'day': 22, 'month': 3, 'year': 2024}},
           'string': 'Sep 29, 2023 to Mar 22, 2024',
           'to': '2024-03-22T00:00:00+00:00'},
 'airing': False,
 'approved': True,
 'background': 'Sousou no Frieren was released on Blu-ray and DVD in seven '
               'volumes from January 24, 2024, to July 17, 2024. The series '
               "aired on Nippon TV's Friday Anime Night block.",
 'broadcast': {'day': 'Fridays',
               'string': 'Fridays at 23:00 (JST)',
               'time': '23:00',
               'timezone': 'Asia/Tokyo'},
 'demographics': [{'mal_id': 27,
                   'name': 'Shounen',
                   'type': 'anime',
                   'url': 'https://myanimelist.net/anime/genre/27/Shounen'}],
 'duration': '24 min per ep',
 'episodes': 28,
 'explicit_genres': [],
 'favorites': 91494,
 'genres': [{'mal_id': 2,
 

### Entidades encontradas

In [33]:
for key in anime.keys():
    print(key)

mal_id
url
images
trailer
approved
titles
title
title_english
title_japanese
title_synonyms
type
source
episodes
status
airing
aired
duration
rating
score
scored_by
rank
popularity
members
favorites
synopsis
background
season
year
broadcast
producers
licensors
studios
genres
explicit_genres
themes
demographics


In [45]:
anime["genres"]

[{'mal_id': 2,
  'type': 'anime',
  'name': 'Adventure',
  'url': 'https://myanimelist.net/anime/genre/2/Adventure'},
 {'mal_id': 46,
  'type': 'anime',
  'name': 'Award Winning',
  'url': 'https://myanimelist.net/anime/genre/46/Award_Winning'},
 {'mal_id': 8,
  'type': 'anime',
  'name': 'Drama',
  'url': 'https://myanimelist.net/anime/genre/8/Drama'},
 {'mal_id': 10,
  'type': 'anime',
  'name': 'Fantasy',
  'url': 'https://myanimelist.net/anime/genre/10/Fantasy'}]

In [35]:
anime["studios"]

[{'mal_id': 11,
  'type': 'anime',
  'name': 'Madhouse',
  'url': 'https://myanimelist.net/anime/producer/11/Madhouse'}]

In [36]:
anime["themes"]

[]

### Testando endpoint individual

In [37]:
response = requests.get(f"{BASE_URL}/anime/10")

print(response.status_code)
print(response.json())

504
{'status': 504, 'type': 'BadResponseException', 'message': 'Jikan failed to connect to MyAnimeList. MyAnimeList may be down/unavailable or refuses to connect', 'error': None}


In [38]:
requests.get(f"{BASE_URL}/anime/1").status_code

200

Conclusão

- Os identificadores não são contínuos.
- Não é adequado construir a ingestão percorrendo mal_id.

### Testando paginação

In [39]:
page = requests.get(
    f"{BASE_URL}/top/anime?page=2"
)

page.status_code

504

In [40]:
page.json()["pagination"]

KeyError: 'pagination'